# 71. RG3→추가 CN7 RH 기록 블라인드 전이 예측
RG3620개만학습하고CN7추가1276개키의정답은열지않는다. 모델·피처·임계값규칙은사전잠금. 중복센서충돌은정답을보지않은입력품질검사에서별도계수한다. 현재샷이전의제품내이력만사용한다. 예측봉인뒤에만72번에서정답개봉한다.
기존CN7가스13개를별도독립이라고재사용하는것이아니다. 새후보키에대해서도과거사용흔적감사와미기록조회가능성을구분해야한다. 제품간검증이지RG3독립미래성공증명이아니다.


In [ ]:
from pathlib import Path
import json,hashlib,datetime
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s68=''.join(json.loads((root/'notebooks/68_rg3_history_normalized_cluster_experts.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s68.split('tail=s67')[0].replace('rolling(100,min_periods=20)','rolling(100,min_periods=5)').replace('rg3_history_ratio_','rg3_cn7transfer_train_'))
train=d.copy();ratio_train=F.copy();yy=y.copy();train_groups=groups.copy()
sensors=pd.read_csv(root/'data/moldset_labeled_cn7.csv',nrows=0).columns.drop(['Unnamed: 0','PassOrFail']).tolist()
raw=pd.read_csv(root/'data/original/labeled_data.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp']+sensors)
raw=raw[raw.PART_NAME.str.contains('CN7',na=False)&raw.PART_NAME.str.contains('RH',na=False)].copy();raw.TimeStamp=pd.to_datetime(raw.TimeStamp)
old=pd.read_csv(root/'data/original_split/moldset_labeled_cn7.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp'])
old=old[old.PART_NAME.str.contains('RH',na=False)].drop_duplicates();old.TimeStamp=pd.to_datetime(old.TimeStamp)
meta=raw[['EQUIP_CD','PART_NAME','TimeStamp']].drop_duplicates().merge(old.assign(known=True),on=['EQUIP_CD','PART_NAME','TimeStamp'],how='left')
meta=meta[meta.known.isna()].drop(columns='known')
assert len(meta)==1276
key=['EQUIP_CD','PART_NAME','TimeStamp']
values=meta.merge(raw,on=key,validate='one_to_many')
ranges=values.groupby(key)[sensors].max()-values.groupby(key)[sensors].min()
conflict=ranges.gt(1e-4).any(axis=1)
safe_keys=conflict[~conflict].reset_index()[key]
current=safe_keys.merge(raw.drop_duplicates(key),on=key,validate='one_to_one')
meta.merge(conflict.rename('sensor_conflict').reset_index(),on=key).to_csv(tables/'cn7_transfer1276_candidate_input_audit.csv',index=False,encoding='utf-8-sig')
print('CN7 extra metadata',len(meta),'sensor-consistent unique inputs',len(current),'conflicting',int(conflict.sum()),flush=True)
assert len(current)>0
current['product']='CN7'
keys=['EQUIP_CD','product','TimeStamp']
idx=pd.MultiIndex.from_frame(current[keys])
observed=observed[~pd.MultiIndex.from_frame(observed[keys]).isin(idx)].copy()
observed=pd.concat([observed,current[keys+sensors].assign(sensor_conflict=False)],ignore_index=True).sort_values(['EQUIP_CD','TimeStamp','product']).reset_index(drop=True)
gap=observed.groupby('EQUIP_CD').TimeStamp.diff().dt.total_seconds()
switch=observed['product'].ne(observed.groupby('EQUIP_CD')['product'].shift())
observed['history_segment']=(switch|gap.gt(1800)|gap.isna()).cumsum()
ratios={}
for c in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']:
 prior=observed.groupby('history_segment')[c].shift()
 pg=prior.groupby(observed.history_segment)
 observed[c+'__past10_std']=pg.transform(lambda a:a.rolling(10,min_periods=5).std())
 observed[c+'__delta1']=observed[c]-prior
 observed[c+'__long_scale']=pg.transform(lambda a:a.rolling(100,min_periods=5).std())
test=current[keys+['PART_NAME']].merge(observed,on=keys,validate='one_to_one')
for c in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']:
 den=test[c+'__long_scale'].where(test[c+'__long_scale']>1e-8)
 ratios[c+'__jitter10_over100']=test[c+'__past10_std']/den
 ratios[c+'__delta_over100']=test[c+'__delta1']/den
ratio_test=pd.DataFrame(ratios).replace([np.inf,-np.inf],np.nan)
stdcols=['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']
memo={}
def fit_predict(ref,query_std,query_ratio):
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(train.iloc[ref][stdcols])
 tr=im.transform(train.iloc[ref][stdcols]);te=im.transform(query_std)
 m=RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=2)
 a=m.fit(tr,yy[ref]).predict_proba(te)[:,1]
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(ratio_train.iloc[ref][yy[ref]==0])
 tr=im.transform(ratio_train.iloc[ref]);te=im.transform(query_ratio)
 sc=RobustScaler().fit(tr[yy[ref]==0]);tr=sc.transform(tr);te=sc.transform(te)
 pos=np.flatnonzero(yy[ref]==1);normal=np.flatnonzero(yy[ref]==0)
 k=min(3,len(np.unique(tr[pos],axis=0)))
 lab=KMeans(n_clusters=k,n_init=20,random_state=42).fit_predict(tr[pos]) if k>1 else np.zeros(len(pos),int)
 outputs=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in [42,7]:
   modes=[]
   for mode in np.unique(lab):
    ids=np.r_[normal,pos[lab==mode]]
    m=cls(n_estimators=100,max_depth=5,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2)
    modes.append(m.fit(tr[ids],yy[ref][ids]).predict_proba(te)[:,1])
   outputs.append(np.max(modes,axis=0))
 return np.column_stack([a,np.mean(outputs,axis=0)])
ref=np.arange(len(train));out=fit_predict(ref,test[stdcols],ratio_test)
inner=np.full((len(train),2),np.nan)
for b in train_groups.unique():
 mask=train_groups.eq(b).to_numpy()
 inner[mask]=fit_predict(ref[~mask],train.iloc[ref[mask]][stdcols],ratio_train.iloc[ref[mask]])
 print('RG3-only calibration block',b,flush=True)
cut=np.quantile(inner[yy==0],.95,axis=0,method='higher')
a=out[:,0]>cut[0];b=out[:,1]>cut[1]
pred=test[['EQUIP_CD','PART_NAME','TimeStamp']].copy()
pred['reference_prediction']=a.astype(int);pred['secondary_prediction']=b.astype(int);pred['primary_prediction']=(a|b).astype(int)
pred['reference_score']=out[:,0];pred['secondary_score']=out[:,1]
pred['reference_threshold']=cut[0];pred['secondary_threshold']=cut[1]
path=tables/'cn7_transfer1276_predictions_SEALED.csv'
assert not path.exists(),'Do not overwrite predictions after label unblinding.'
pred.to_csv(path,index=False,encoding='utf-8-sig')
protocol=root/'reports/cn7_extra1276_locked_transfer_protocol.json'
pd.DataFrame([dict(sealed_at_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),prediction_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),protocol_sha256=hashlib.sha256(protocol.read_bytes()).hexdigest(),CN7_extra_labels_loaded=False,RG3_train_shots=len(train),test_shots=len(pred),sensor_conflicting_keys=int(conflict.sum()))]).to_csv(tables/'cn7_transfer1276_prediction_seal.csv',index=False,encoding='utf-8-sig')
print('CN7 predictions sealed before extra CN7 labels opened.')
print(pred[['primary_prediction','reference_prediction','secondary_prediction']].sum().to_string())
